# Deb Stone public-record lead pipeline

Run this notebook to create reviewable address leads, write a CSV, and update the local SQLite lead database. It uses public sources only and preserves source URLs and timestamps.

The Contra Costa parcel download supplies addresses and APNs, **not owner names**. Do not add names unless an approved official public-record source visibly provides one.

## One-time setup

If this environment has not installed the scraper dependencies, uncomment and run the next line.

In [ ]:
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import sys

# Allows the notebook to run from either the project root or scraper folder.
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'scraper').exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif (PROJECT_ROOT / 'scrape_public_data.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    raise RuntimeError('Open this notebook from the eastbayrealestate project or its scraper folder.')

from scraper.scrape_public_data import (
    build_records_from_parcel_url,
    build_records_from_target,
    read_queries,
    save_results,
    save_to_database,
)

## Configure the run

Set `RUN_MODE` to `parcel` for automatic address/APN generation, `queries` for an address list, or `targets` for the reviewed target registry. Update the parcel ZIP URL from the County GIS download directory before each new monthly run.

In [ ]:
RUN_MODE = 'parcel'  # parcel | queries | targets

# Verified public archive used for the initial run. Replace with the current monthly ZIP when available.
PARCEL_ZIP_URL = 'https://gis.cccounty.us/Downloads/Assessor/Parcels_Public_June2026.zip'
TARGET_CITIES = ['Walnut Creek', 'Lafayette', 'Orinda', 'Pleasant Hill', 'Concord']
LEAD_LIMIT = 250

QUERIES_FILE = PROJECT_ROOT / 'scraper' / 'addresses.txt'
TARGETS_FILE = PROJECT_ROOT / 'scraper' / 'contra_costa_targets.json'
OUTPUT_CSV = PROJECT_ROOT / 'scraper' / 'output' / 'parcel_leads.csv'
DATABASE_PATH = PROJECT_ROOT / 'scraper' / 'output' / 'leads.sqlite3'

print(f'Mode: {RUN_MODE}')
print(f'CSV: {OUTPUT_CSV}')
print(f'Database: {DATABASE_PATH}')

In [ ]:
import json

if RUN_MODE == 'parcel':
    records = build_records_from_parcel_url(
        PARCEL_ZIP_URL,
        cities=TARGET_CITIES,
        limit=LEAD_LIMIT,
    )
elif RUN_MODE == 'queries':
    records = []
    for query in read_queries(str(QUERIES_FILE)):
        records.extend(build_records_from_target({'type': 'property_search', 'query': query}))
elif RUN_MODE == 'targets':
    targets = json.loads(TARGETS_FILE.read_text(encoding='utf-8')).get('sources', [])
    records = []
    for target in targets:
        records.extend(build_records_from_target(target))
else:
    raise ValueError("RUN_MODE must be 'parcel', 'queries', or 'targets'")

print(f'Collected {len(records):,} raw record(s).')

## Save and review

This writes both outputs. Re-running updates existing leads by normalized address and city, preserves a non-default review status, and appends a new source observation.

In [ ]:
csv_path = save_results(records, str(OUTPUT_CSV), limit=LEAD_LIMIT)
saved_leads = save_to_database(records, str(DATABASE_PATH), limit=LEAD_LIMIT)
print(f'Saved {saved_leads:,} lead(s) to {csv_path}')
print(f'Updated database: {DATABASE_PATH}')

In [ ]:
import csv
from collections import Counter

with OUTPUT_CSV.open(encoding='utf-8', newline='') as file:
    leads = list(csv.DictReader(file))

for lead in leads[:20]:
    print(f"{lead['lead_rating']} | {lead['address']}, {lead['city']} | APN {lead['parcel_number']} | {lead['review_status']}")

print('\nLead count by city:')
for city, count in Counter(lead['city'] for lead in leads).most_common():
    print(f'- {city}: {count}')

## Review rules

- Treat every result as a research candidate, not an automatic outreach target.
- Verify the source URL before acting on a lead.
- Keep names blank for parcel-only records.
- Record outreach decisions and any do-not-contact request in the database before another run.